# packages

In [1]:
import os
from glob import glob
import re
import numpy as np
import cv2
from PIL import Image
import argparse
import shutil
import matplotlib.pyplot as plt
from utils import *

# code run only once

## video to frame

In [7]:
# Call the function to convert the video to images
convert_video_to_images('src/drill/1-1.mp4', 'drill1-1')

Output folder already exists: drill1-1
Recreating output folder: drill1-1
Total frames: 238


In [8]:
convert_video_to_images('src/drill/1-2.mp4', 'drill1-2')

Creating output folder: drill1-2
Total frames: 238


In [9]:
convert_video_to_images('src/drill/1-3.mp4', 'drill1-3')

Creating output folder: drill1-3
Total frames: 237


## functions

In [2]:
kernel=np.array((9,9), dtype=np.uint8)

The GIF of benchmark video (1-1)

In [10]:
create_gif_from_images('src.GIF', 'drill1-1', '.jpg')

start to create GIF


In [5]:
def fd(src, new):
    # src: name of folder
    # new: name of folder
    src_paths = sorted(glob(f"{src}/*.jpg"), key=lambda x: int(re.search(r'(\d+).jpg', os.path.basename(x)).group(1)))
    new_paths = sorted(glob(f"{new}/*.jpg"), key=lambda x: int(re.search(r'(\d+).jpg', os.path.basename(x)).group(1)))
    
    thresh = 1000
    num_frames = 237
    box_result = []
    for idx in range(0, num_frames):
        # read frames
        frame1_bgr = cv2.imread(src_paths[idx])
        frame2_bgr = cv2.imread(new_paths[idx])

        # get detections
        detections = get_detections(cv2.cvtColor(frame1_bgr, cv2.COLOR_BGR2GRAY), 
                                    cv2.cvtColor(frame2_bgr, cv2.COLOR_BGR2GRAY), 
                                    bbox_thresh=thresh,
                                    nms_thresh=1e-4)
        box_result.append(detections)
    print('the box thresh: ', thresh)
    
    ####visualize
    if not os.path.exists('temp'):
        os.makedirs('temp')
    else:
        shutil.rmtree('temp')
        os.makedirs('temp')
        
    print('start to visualize the box on the images')

    for idx in range(0, num_frames):
        # read frames
        frame_bgr = cv2.imread(new_paths[idx])
        detections = box_result[idx]                           
        # draw bounding boxes on frame
        draw_bboxes(frame_bgr, detections)

        # save image for GIF
        fig = plt.figure(figsize=(1280/100, 720/100)) # (15, 7)  / (1280/100, 720/100)
        plt.imshow(frame_bgr)
        plt.axis('off')
        fig.savefig(f"temp/frame_{idx}.png")
        plt.close()

    file_path = f"{new}.GIF"

    # Check if the file exists before attempting to delete it
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"{file_path} has been deleted.")
    else:
        print(f"{file_path} does not exist.")

    create_gif_from_images(file_path, 'temp', '.png')

# code to execute

In [6]:
fd('drill1-1', 'drill1-2') #the result is saved in drill1-3.GIF
fd('drill1-1', 'drill1-3') #the result is saved in drill1-3.GIF

the box thresh:  1000
start to visualize the box on the images
drill1-2.GIF does not exist.
start to create GIF
the box thresh:  1000
start to visualize the box on the images
drill1-3.GIF does not exist.
start to create GIF
